In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("LOSS CALCULATION REPORT")
print("="*60)

# ============================================
# MODEL 1: Telco Churn
# ============================================
print("\n📊 MODEL 1: Telco Churn (Stacking)")
print("-"*40)

telco = pd.read_csv(r'C:\Users\user\DSS_Project\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv')
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce').fillna(0)
telco['churn'] = (telco['Churn'] == 'Yes').astype(int)

le = LabelEncoder()
cat_cols = [c for c in telco.select_dtypes(include='object').columns if c != 'Churn']
for col in cat_cols:
    telco[col] = le.fit_transform(telco[col].astype(str))

features = [c for c in telco.columns if c not in ['Churn','churn','customerID']]
X = telco[features]
y = telco['churn']

smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X, y)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_bal)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_bal, test_size=0.2, random_state=42)

estimators = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=200, random_state=42))
]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5, n_jobs=1)
stack.fit(X_train, y_train)

y_proba_telco = stack.predict_proba(X_test)
y_pred_telco  = stack.predict(X_test)

acc_telco    = stack.score(X_test, y_test)
loss_telco   = log_loss(y_test, y_proba_telco)
brier_telco  = brier_score_loss(y_test, y_proba_telco[:,1])

print(f"   Accuracy     : {acc_telco*100:.2f}%")
print(f"   Log Loss     : {loss_telco:.4f}")
print(f"   Brier Score  : {brier_telco:.4f}")
print(f"   Loss Rating  : {'✅ Good' if loss_telco < 0.4 else '⚠️ Average' if loss_telco < 0.6 else '❌ High'}")

LOSS CALCULATION REPORT

📊 MODEL 1: Telco Churn (Stacking)
----------------------------------------
   Accuracy     : 85.80%
   Log Loss     : 0.3321
   Brier Score  : 0.1017
   Loss Rating  : ✅ Good


In [10]:
# ============================================
# MODEL 2: Bank Churn
# ============================================
print("\n📊 MODEL 2: Bank Churn (Stacking)")
print("-"*40)

bank = pd.read_csv(r'C:\Users\user\DSS_Project\data\raw\Churn_Modelling.csv')

le2 = LabelEncoder()
bank['Geography'] = le2.fit_transform(bank['Geography'])
bank['Gender']    = le2.fit_transform(bank['Gender'])

features_bank = ['CreditScore','Geography','Gender','Age','Tenure',
                 'Balance','NumOfProducts','HasCrCard',
                 'IsActiveMember','EstimatedSalary']

X_bank = bank[features_bank]
y_bank = bank['Exited']

smote2 = SMOTE(random_state=42)
X_bal2, y_bal2 = smote2.fit_resample(X_bank, y_bank)
scaler2 = StandardScaler()
X_scaled2 = scaler2.fit_transform(X_bal2)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_scaled2, y_bal2, test_size=0.2, random_state=42)

estimators2 = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=200, random_state=42))
]
stack2 = StackingClassifier(
    estimators=estimators2,
    final_estimator=LogisticRegression(),
    cv=5, n_jobs=1)
stack2.fit(X_train2, y_train2)

y_proba_bank = stack2.predict_proba(X_test2)
y_pred_bank  = stack2.predict(X_test2)

acc_bank   = stack2.score(X_test2, y_test2)
loss_bank  = log_loss(y_test2, y_proba_bank)
brier_bank = brier_score_loss(y_test2, y_proba_bank[:,1])

print(f"   Accuracy     : {acc_bank*100:.2f}%")
print(f"   Log Loss     : {loss_bank:.4f}")
print(f"   Brier Score  : {brier_bank:.4f}")
print(f"   Loss Rating  : {'✅ Good' if loss_bank < 0.4 else '⚠️ Average' if loss_bank < 0.6 else '❌ High'}")

# ============================================
# MODEL 3: Marketing ROI
# ============================================
print("\n📊 MODEL 3: Marketing Campaign Success")
print("-"*40)

mkt = pd.read_csv(r'C:\Users\user\DSS_Project\data\clean\marketing_clean.csv')

camp_features = ['marketing_budget_usd', 'discount_percentage',
                 'num_promotions', 'website_traffic',
                 'email_open_rate', 'conversion_rate']

X_camp = mkt[camp_features].fillna(0)
y_camp = (mkt['sales_revenue_usd'] > mkt['sales_revenue_usd'].median()).astype(int)

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X_camp, y_camp, test_size=0.2, random_state=42)

camp_model = GradientBoostingClassifier(
    n_estimators=200, random_state=42)
camp_model.fit(X_train3, y_train3)

y_proba_camp = camp_model.predict_proba(X_test3)
y_pred_camp  = camp_model.predict(X_test3)

acc_camp   = camp_model.score(X_test3, y_test3)
loss_camp  = log_loss(y_test3, y_proba_camp)
brier_camp = brier_score_loss(y_test3, y_proba_camp[:,1])

print(f"   Accuracy     : {acc_camp*100:.2f}%")
print(f"   Log Loss     : {loss_camp:.4f}")
print(f"   Brier Score  : {brier_camp:.4f}")
print(f"   Loss Rating  : {'✅ Good' if loss_camp < 0.4 else '⚠️ Average' if loss_camp < 0.6 else '❌ High'}")

# ============================================
# FINAL SUMMARY
# ============================================
print("\n" + "="*60)
print("FINAL LOSS SUMMARY")
print("="*60)
print(f"{'Model':<30} {'Accuracy':>10} {'Log Loss':>10} {'Brier':>10} {'Rating':>10}")
print("-"*60)
print(f"{'Telco Churn (Stacking)':<30} {'85.00%':>10} {'0.3321':>10} {'0.1017':>10} {'✅ Good':>10}")
print(f"{'Bank Churn (Stacking)':<30} {acc_bank*100:>9.2f}% {loss_bank:>10.4f} {brier_bank:>10.4f} {'✅ Good' if loss_bank < 0.4 else '⚠️ Avg':>10}")
print(f"{'Marketing Campaign':<30} {acc_camp*100:>9.2f}% {loss_camp:>10.4f} {brier_camp:>10.4f} {'✅ Good' if loss_camp < 0.4 else '⚠️ Avg':>10}")
print("="*60)
print("\nLog Loss Scale:")
print("   < 0.3  = Excellent ✅✅")
print("   < 0.4  = Good      ✅")
print("   < 0.6  = Average   ⚠️")
print("   > 0.6  = High      ❌")


📊 MODEL 2: Bank Churn (Stacking)
----------------------------------------
   Accuracy     : 86.32%
   Log Loss     : 0.3235
   Brier Score  : 0.0988
   Loss Rating  : ✅ Good

📊 MODEL 3: Marketing Campaign Success
----------------------------------------
   Accuracy     : 68.44%
   Log Loss     : 0.5761
   Brier Score  : 0.1994
   Loss Rating  : ⚠️ Average

FINAL LOSS SUMMARY
Model                            Accuracy   Log Loss      Brier     Rating
------------------------------------------------------------
Telco Churn (Stacking)             85.00%     0.3321     0.1017     ✅ Good
Bank Churn (Stacking)              86.32%     0.3235     0.0988     ✅ Good
Marketing Campaign                 68.44%     0.5761     0.1994     ⚠️ Avg

Log Loss Scale:
   < 0.3  = Excellent ✅✅
   < 0.4  = Good      ✅
   < 0.6  = Average   ⚠️
   > 0.6  = High      ❌


In [8]:
import sys
import subprocess

# تثبيت كل المكتبات المطلوبة
packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "imbalanced-learn"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# imports
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print("All libraries installed and imported successfully")

All libraries installed and imported successfully


In [3]:
import sys
import subprocess

# قائمة المكتبات المطلوبة
packages = [
    "pandas",
    "numpy",
    "scikit-learn"
]

# تثبيت أي مكتبة ناقصة
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# الاستيراد
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, brier_score_loss

# تأكيد إن كل حاجة شغالة
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

import sklearn
print("Scikit-learn:", sklearn.__version__)

Pandas: 3.0.2
NumPy: 2.4.4
Scikit-learn: 1.8.0


In [2]:
import sys
import subprocess

# تثبيت pandas لو مش موجودة
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])

# استيراد المكتبة
import pandas as pd

# التأكد إنها اشتغلت
print("Pandas version:", pd.__version__)

Pandas version: 3.0.2
